In [ ]:
import numpy as np
import xarray as xr
import cftime
import pandas as pd
import os

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

TC_path = r'../../../study/data/TC/D4PDF/101.D4PDF_ensembles_Japan/'
TCG_path = r'../../../study/data/TC/D4PDF/TCG-4K/'

lat = np.arange(-90, 91, 1)
lon = np.arange(0, 361, 1)

In [ ]:
exp = ['CC', 'GF', 'HA', 'MI', 'MP', 'MR']

In [ ]:
time = pd.date_range('1951-01', '2011-12', freq='MS')
nt = len(time)
nx, ny = len(lon), len(lat)

ds_tc = xr.Dataset(
    {
        'tct':(['exp', 'time', 'lat', 'lon'], np.zeros((len(exp), nt, ny, nx))),
        'tcg':(['exp', 'time', 'lat', 'lon'], np.zeros((len(exp), nt, ny, nx))),
    },
    coords={
        'exp':(['exp'], exp),
        'time':(['time'], time),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
tct = np.zeros((len(exp), nt, ny, nx))
tcg = np.zeros((len(exp), nt, ny, nx))
for e in range(6):
    for i in range(15):
        file_name = TC_path+rf'xytrackk319b_HFB_4K_{exp[e]}_m{101+i:3d}_2051-2110.txt'
        tc = np.loadtxt(file_name, usecols=[2, 3, 6, 7]).astype(np.int64)
        its = (tc[:, 0] - 2051)*12 + tc[:, 1] - 1
        xls = tc[:, 2] - 10
        xus = tc[:, 2] + 10
        yls = tc[:, 3] + 90 - 5
        yus = tc[:, 3] + 90 + 5
        for j in range(tc.shape[0]):
            tct[e, its[j], yls[j]:yus[j], xls[j]:xus[j]] += 1
for e in range(6):
    for i in range(15):
        file_name = TCG_path+f'Genesis_4K_{exp[e]}_{101+i:3d}.txt'
        tc = np.loadtxt(file_name, usecols=[2, 3, 6, 7], delimiter=',').astype(np.int64)
        its = (tc[:, 0] - 2051)*12 + tc[:, 1] - 1
        xls = tc[:, 2] - 10
        xus = tc[:, 2] + 10
        yls = tc[:, 3] + 90 - 5
        yus = tc[:, 3] + 90 + 5
        for j in range(tc.shape[0]):
            tcg[e, its[j], yls[j]:yus[j], xls[j]:xus[j]] += 1

ds_tc.tct[:] = tct / 15
ds_tc.tcg[:] = tcg / 15
ds_tc.to_netcdf('./data-d4pdf/D4PDF_4K_TC_grid.nc')

In [ ]:
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
start, end = np.datetime64('1950-01'), np.datetime64('2009-01')
cp = cp[np.logical_and(cp > start, cp < end)]
ep = ep[np.logical_and(ep > start, ep < end)]
mix = mix[np.logical_and(mix > start, mix < end)]
suc = suc[np.logical_and(suc > start, suc < end)]
events = [mix, cp, ep, suc]

ds_tc = xr.open_dataset('./data-d4pdf/D4PDF_4K_TC_grid.nc')
tcg_grid = ds_tc.tcg
tct_grid = ds_tc.tct
ssta = xr.open_dataset('./data/ssta.nc').sst.loc['1951':'2011']

rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()

tcg_grid -= tcg_grid.groupby('time.month').map(rolling)
tct_grid -= tct_grid.groupby('time.month').map(rolling)
ssta -= ssta.groupby('time.month').map(rolling)

mon = np.arange(-6, 30)
nt, ny, nx = len(mon), len(lat), len(lon)

ds_nino = xr.Dataset(
    {
        'tcg':(['exp', 'type', 'month', 'lat', 'lon'], np.empty((6, 4, nt, ny, nx))),
        'tct':(['exp', 'type', 'month', 'lat', 'lon'], np.empty((6, 4, nt, ny, nx))),
        'sst':(['type', 'month', 'lat', 'lon'], np.empty((4, nt, ny, nx))),
    },
    coords={
        'exp':(['exp'], exp),
        'type':(['type'], ['MIX', 'CP', 'EP', 'SUC']),
        'month':(['month'], mon),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
for e in range(6):
    for i in range(4):
        event = events[i]
        for j in range(nt):
            ds_nino.tcg[e, i, j] = tcg_grid[e].loc[(event + np.timedelta64(mon[j], "M")).astype('datetime64[ns]')].mean(dim='time').values
            ds_nino.tct[e, i, j] = tct_grid[e].loc[(event + np.timedelta64(mon[j], "M")).astype('datetime64[ns]')].mean(dim='time').values

for i in range(4):
    event = events[i]
    for j in range(nt):
        ds_nino.sst[i, j] =     ssta.loc[(events[i] + np.timedelta64(mon[j], "M")).astype('datetime64[ns]')].mean(dim='time').values
ds_nino.to_netcdf('./data-d4pdf/composite_tc_sst_4K.nc')




In [ ]:
time = pd.date_range('1951-01', '2011-12', freq='MS')
nt = len(time)
nx, ny = len(lon), len(lat)
rolling = lambda x: x.rolling(time=30, min_periods=1, center=True).mean()
def process_space(ssta):
    ssta -= ssta.groupby('time.month').map(rolling)
    ssta_sum = ssta.sel(time=ssta.time.dt.month.isin([6, 7, 8, 9, 10, 11])).coarsen(time=6).mean()
    ssta_win = ssta[11:-1].sel(time=ssta[11:-1].time.dt.month.isin([1, 2, 3, 4, 5, 12])).coarsen(time=6).mean()
    return ssta, ssta_sum, ssta_win
ds_tc = xr.Dataset(
    {
        'tct_grid':(['time', 'lat', 'lon'], np.zeros((nt, ny, nx))),
    },
    coords={
        'time':(['time'], time),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
output_path = r'/Volumes/shiki/study/data/TC/D4PDF/'
tct_grid = np.zeros((nt, ny, nx))
for i in range(1, 101):
    file_name = TC_path+rf'xytrackk319b_HPB_m{i:03d}_1951-2011.txt'
    output_name =     output_path + rf'TCT_ano/tct_{i:03d}.nc'
    output_name_sum = output_path + rf'TCT_sum/tct_{i:03d}.nc'
    output_name_win = output_path + rf'TCT_win/tct_{i:03d}.nc'
    tct = np.zeros((nt, ny, nx))
    print(output_name, output_name_sum, output_name_win)
    tc = np.loadtxt(file_name, usecols=[2, 3, 6, 7]).astype(np.int64)
    its = (tc[:, 0] - 1951)*12 + tc[:, 1] - 1
    xls = tc[:, 2] - 10
    xus = tc[:, 2] + 10
    yls = tc[:, 3] + 90 - 5
    yus = tc[:, 3] + 90 + 5
    for j in range(tc.shape[0]):
        tct[its[j], yls[j]:yus[j], xls[j]:xus[j]] += 1
    tct_grid[:] += tct[:]
    ds = xr.Dataset({'tcta_grid':(['time', 'lat', 'lon'], tct)}, coords=ds_tc.coords)
    tcta, tcta_sum, tcta_win = process_space(ds.tcta_grid)
    ds_sum = xr.Dataset({'tcta_sum':tcta_sum}, coords=tcta_sum.coords)
    ds_win = xr.Dataset({'tcta_win':tcta_win}, coords=tcta_win.coords)
    ds.tcta_grid[:] = tcta
    ds.to_netcdf(output_name)
    ds_sum.to_netcdf(output_name_sum)
    ds_win.to_netcdf(output_name_win)
ds_tc.tct_grid[:] = tct_grid/100
ds_tc.to_netcdf('/Volumes/shiki/study/resherch/ENSO develop/data-d4pdf/D4PDF_TCT_grid.nc')

In [ ]:
output_path = r'/Volumes/shiki/study/data/TC/D4PDF/'
ds = xr.open_dataset(output_path+f'TCT_ano/tct_001.nc')
tct = ds.tcta_grid.loc[:, -40:40, 0:359]
month = tct.time.dt.month
lat_nh = (tct.lat >= 5) & (tct.lat <= 25) 
lat_sh = (tct.lat >= -25) & (tct.lat <= -5)
is_jun_nov = (month >= 6) & (month <= 11)
is_dec_may = (month >= 12) | (month <= 5)
mask = (is_jun_nov & lat_nh) | (is_dec_may & lat_sh)

for i in range(1, 101):
    TCT_path_grid = output_path
    ds = xr.open_dataset(output_path+f'/TCT_ano/tct_{i:03d}.nc')
    tct = ds.tcta_grid.loc[:, -40:40, 0:359]
    # tct = tct.groupby('time.month') - tct.groupby('time.month').mean()
    tct_hov = tct.where(mask).mean('lat')
    tct_hov = tct_hov.assign_coords(
                year=tct_hov.time.dt.year,
                month=tct_hov.time.dt.month
            ).set_index(time=["year", "month"]).unstack("time").transpose('year', 'month', 'lon')
    ds_hov = xr.Dataset({'tct_hov':tct_hov}, coords=tct_hov.coords)
    output_name = output_path+rf'TCT_hov/tct_hov_{i:03d}.nc'
    print(output_name)
    ds_hov.to_netcdf(output_name)